# 11 — Multi-agent: one job per agent

**Both** — steer who answers, and scope what each agent sees and spends.

When you finish you can:

- hand a sub-task to a helper agent from a tool, on one shared bill and one budget
- build a router whose model must pick a specialist from a menu you wrote
- route in plain Python with a classifier and `match`
- pick `instructions` over `system_prompt` when a conversation moves between agents

> Most cells here use your `MODEL`, so they need your key. Only two run **keyless**: the budget cell and the
> hand-off cell use scripted models (from 08).

In [ ]:
from learningdantic import setup

MODEL = setup()

## Why more than one agent?

An agent is a model + instructions + tools + an output type. When one agent does every job, its instructions
grow long and its tools pile up. Split the work and each agent gets one job: short instructions (**steer**)
and only the tools and data that job needs (**scope**).

The real question is **who decides which agent runs next**:

| Who decides | Pattern | Does control come back? |
|---|---|---|
| The model, by calling a tool | Delegation | Yes: the first agent keeps talking |
| The model, by picking an output function | Router | No: the specialist's answer is final |
| Your Python code | Programmatic hand-off | You decide |

Docs: [Multi-agent applications](https://pydantic.dev/docs/ai/guides/multi-agent-applications/).

## Delegation: a tool that runs another agent

The `desk` agent talks to the customer. Only `policy_expert` knows the refund policy: it is in its
instructions and nowhere else. `desk` asks it through a tool, gets the answer back, and keeps talking.

The tool passes three things on:

- `deps=ctx.deps`: the helper sees the same customer.
- `usage=ctx.usage`: the helper's requests go on the same bill, so `result.usage` counts both agents.
- `usage_limits=ctx.usage_limits`: the helper obeys the same budget. The next cell shows why.

`name="desk"` gives each agent a label for traces. Without it, Pydantic AI guesses one from your variable names.

> ⚡ **The async rule, applied to a tool.** `ask_policy` has `await policy_expert.run(...)` inside, so it is
> `async def`. You never call a tool yourself (Pydantic AI awaits it), and tools with no agent run inside stay
> plain `def`.

In [ ]:
from dataclasses import dataclass

from pydantic_ai import Agent, RunContext


@dataclass
class Customer:
    name: str
    plan: str


policy_expert = Agent(
    MODEL,
    name="policy_expert",
    deps_type=Customer,
    instructions=(
        "Refund policy: monthly plans get a full refund within 14 days of a charge. "
        "Annual plans get a pro-rated refund. Gift cards are never refunded. "
        "Answer in one sentence."
    ),
)


@policy_expert.instructions
def plan_note(ctx: RunContext[Customer]) -> str:
    return f"The customer is on the {ctx.deps.plan} plan."


desk = Agent(
    MODEL,
    name="desk",
    deps_type=Customer,
    instructions=(
        "You are a support desk. For any refund question, call ask_policy first. "
        "Then answer in two short sentences and use the customer's name."
    ),
)


@desk.instructions
def customer_note(ctx: RunContext[Customer]) -> str:
    return f"The customer's name is {ctx.deps.name}."


@desk.tool
async def ask_policy(ctx: RunContext[Customer], question: str) -> str:
    """Ask the refund-policy expert. Use it for every refund question."""
    answer = await policy_expert.run(
        question, deps=ctx.deps, usage=ctx.usage, usage_limits=ctx.usage_limits
    )
    return answer.output


result = await desk.run(
    "Can I get my money back? I paid last week.", deps=Customer("Ada", "monthly")
)
print(result.output)
print("requests on one bill:", result.usage.requests)

> ✅ **Checkpoint** — two short sentences to Ada that mention the 14-day refund window, which only
> `policy_expert` knew. Then `requests on one bill: 3`: `desk` asked its model twice (once to call the tool,
> once to answer) and `policy_expert` once. Without `usage=ctx.usage` the count would say 2, and the helper's
> spend would be missing from the bill.

> 🔭 **In Logfire** — every run span carries its agent's `name=`. Open the `desk` run and you'll find the
> `policy_expert` run nested inside the `ask_policy` tool call, with its own tokens.

## One budget for the whole team

In notebook 10 you capped a run with `UsageLimits`; going over raises `UsageLimitExceeded`. A helper run
started inside a tool is a separate run, so it only obeys the limits you hand it.

This cell proves it offline with scripted models (`FunctionModel`, from 08). Two bosses run the same script
under the same cap of 1 request; only the second forwards `usage_limits`. We pass in our own `RunUsage()`
(an empty bill) so we can read it after the error. These agents have no deps, so their tools take plain
`ctx: RunContext`, with no `[...]`.

`(await helper.run(...)).output` runs the helper and takes its answer in one line: the brackets make the
run finish before `.output` is read.

In [ ]:
from pydantic_ai import RunUsage, UsageLimitExceeded, UsageLimits
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart
from pydantic_ai.models.function import FunctionModel


def call_tool_then_answer(messages, info):
    """Scripted boss: first call its one tool, then answer once the result is back."""
    if len(messages) == 1:
        return ModelResponse(
            parts=[ToolCallPart(info.function_tools[0].name, {"question": "Refund?"})]
        )
    return ModelResponse(parts=[TextPart("Here is the policy.")])


def helper_model(messages, info):
    """Scripted helper: always answers straight away."""
    return ModelResponse(parts=[TextPart("14 days.")])


helper = Agent(FunctionModel(helper_model), name="helper")

leaky_boss = Agent(FunctionModel(call_tool_then_answer), name="leaky_boss")


@leaky_boss.tool
async def ask_helper(ctx: RunContext, question: str) -> str:
    return (await helper.run(question, usage=ctx.usage)).output  # forgot usage_limits


safe_boss = Agent(FunctionModel(call_tool_then_answer), name="safe_boss")


@safe_boss.tool
async def ask_helper_safely(ctx: RunContext, question: str) -> str:
    return (await helper.run(question, usage=ctx.usage, usage_limits=ctx.usage_limits)).output


for boss in (leaky_boss, safe_boss):
    bill = RunUsage()
    try:
        await boss.run("Can I get a refund?", usage=bill, usage_limits=UsageLimits(request_limit=1))
    except UsageLimitExceeded:
        print(f"{boss.name}: stopped after {bill.requests} request(s) on a cap of 1")

> ✅ **Checkpoint** — exactly these two lines:
>
> ```
> leaky_boss: stopped after 2 request(s) on a cap of 1
> safe_boss: stopped after 1 request(s) on a cap of 1
> ```
>
> `leaky_boss` spent 2 requests on a cap of 1: its helper ran past the budget, and the error only came on the
> boss's next turn. `safe_boss` stopped before the helper spent anything. Forward `usage_limits=ctx.usage_limits`
> every time you delegate.

## A router: the model must pick from your menu

Put functions in `output_type` and the router's model must end its run by calling exactly one of them.
It cannot answer in free text. The function's return value becomes the run's output, and the router never
speaks again.

- Each **docstring is a routing rule**: the model reads it to choose. Your steering lives there.
- A `BaseModel` in the list (here `Escalate`) is a plain structured answer with no specialist behind it.
- Like a tool, an output function can take `ctx: RunContext` first (Pydantic AI fills it in), and the model
  fills in the other arguments, here `question`.
- `to_billing` and `to_tech` await an agent run, so they are `async def`, like `ask_policy` above.

In [ ]:
from pydantic import BaseModel

billing_agent = Agent(
    MODEL, name="billing", instructions="You are billing support. Answer in at most two sentences."
)
tech_agent = Agent(
    MODEL,
    name="tech",
    instructions="You are technical support. Give one concrete next step in at most two sentences.",
)


class Escalate(BaseModel):
    """Use when the message is not about billing or tech, or the customer asks for a human."""

    reason: str


async def to_billing(ctx: RunContext, question: str) -> str:
    """Invoices, charges, refunds, plans and prices."""
    answer = await billing_agent.run(question, usage=ctx.usage, usage_limits=ctx.usage_limits)
    return f"[billing] {answer.output}"


async def to_tech(ctx: RunContext, question: str) -> str:
    """Bugs, error messages, logins, crashes and setup problems."""
    answer = await tech_agent.run(question, usage=ctx.usage, usage_limits=ctx.usage_limits)
    return f"[tech] {answer.output}"


router = Agent(
    MODEL,
    name="router",
    output_type=[to_billing, to_tech, Escalate],
    instructions="Send the customer's message to the right desk. Pass the message on word for word.",
)

for message in [
    "I was charged twice this month.",
    "The app crashes when I upload a photo.",
    "Can I speak to a manager, please?",
]:
    routed = await router.run(message)
    print(f"{message}\n  -> {routed.output!r}")

> ✅ **Checkpoint** — the first message comes back as `'[billing] …'`, the second as `'[tech] …'`, and
> the third as `Escalate(reason='…')`: a structured answer with no specialist behind it. The router itself never
> wrote a word to the customer. Your wording will differ; the routes should not.

### What the router's model actually sees

Same habit as 08: swap in a spy model and print what it was offered. `info.output_tools` is the menu, and
`info.allow_text_output` says whether a plain-text reply is allowed. From a list, each choice is named
`final_result_<name>`. Wrap a choice in `ToolOutput(..., name=...)` to give it a clearer name for traces and
scripted tests.

In [ ]:
from pydantic_ai import ToolOutput


def spy(messages, info):
    """Print the menu, then pick the last item (Escalate) so no specialist runs."""
    for tool in info.output_tools:
        print(f"  {tool.name}: {tool.description}")
    print("  free text allowed:", info.allow_text_output)
    return ModelResponse(
        parts=[ToolCallPart(info.output_tools[-1].name, {"reason": "just looking"})]
    )


with router.override(model=FunctionModel(spy)):
    await router.run("anything")

renamed = Agent(
    FunctionModel(spy),
    name="renamed_router",
    output_type=[
        ToolOutput(to_billing, name="billing"),
        ToolOutput(to_tech, name="tech"),
        ToolOutput(Escalate, name="escalate"),
    ],
)
spied = await renamed.run("anything")
print(repr(spied.output))

> ✅ **Checkpoint** — two menus. Each choice's description is its docstring, which is what the model routes
> by, and `free text allowed: False` both times. The first menu uses the `final_result_` names:
>
> ```
>   final_result_to_billing: Invoices, charges, refunds, plans and prices.
>   final_result_to_tech: Bugs, error messages, logins, crashes and setup problems.
>   final_result_Escalate: Use when the message is not about billing or tech, or the customer asks for a human.
>   free text allowed: False
> ```
>
> The second shows `billing`, `tech` and `escalate` from `ToolOutput`. The last line is the spy's pick:
> `Escalate(reason='just looking')`.

## Programmatic hand-off: your code decides

Sometimes you don't want a model choosing the next step at all. Ask a small classifier for a label, then let
Python pick the agent. Its `output_type` is a `Literal`: a type hint that allows only the listed strings, so
the label is always one of them.

- `match kind:` compares `kind` with each `case` in order; `case _:` means "anything else".
- One `RunUsage()` and one `UsageLimits` go to every run: one bill, one budget, like the tool above.
- Same rule as `ask_policy`: `handle` has `await` inside, so it is `async def`. This time you call it
  yourself, so you write `await handle(...)`.

In [ ]:
from typing import Literal

classifier = Agent(
    MODEL,
    name="classifier",
    output_type=Literal["billing", "tech", "other"],
    instructions=(
        "Label the support message: billing (money, invoices, plans), "
        "tech (bugs, errors, logins) or other."
    ),
)


async def handle(message: str) -> str:
    bill = RunUsage()
    budget = UsageLimits(request_limit=3)
    kind = (await classifier.run(message, usage=bill, usage_limits=budget)).output
    match kind:
        case "billing":
            answer = (await billing_agent.run(message, usage=bill, usage_limits=budget)).output
        case "tech":
            answer = (await tech_agent.run(message, usage=bill, usage_limits=budget)).output
        case _:
            answer = "Passed to a human."
    return f"[{kind}] {answer} | requests: {bill.requests}"


print(await handle("My card was charged twice."))
print(await handle("Login fails with 'invalid token'."))
print(await handle("What's your office address?"))

> ✅ **Checkpoint** — three lines ending in `| requests: 2`, `| requests: 2` and `| requests: 1`: a
> `[billing]` answer, a `[tech]` answer, and `[other] Passed to a human.` (only the classifier ran). The
> router would also spend 2 for billing or tech (router + specialist), but the route is now a line of Python
> you can read, test and log.

## Handing a conversation to another agent

To let the next agent see what the customer already said, pass the first run's messages:
`message_history=first.all_messages()`. The two ways to write an agent's role behave differently here:

- `system_prompt=` is saved **inside** the history as a message part, so it travels with the conversation.
  Also, whenever you pass `message_history`, the next agent's own `system_prompt` is not added at all, even
  if the history has none.
- `instructions=` are written on each saved request only as a record of what was sent. A new run ignores the
  old ones and sends its own agent's instructions, worked out fresh.

So an agent whose role lives in `system_prompt` can silently lose it when it receives a history. Write roles
as `instructions`, as this course does: the role stays with the agent, not the conversation.

In [ ]:
from pydantic_ai.messages import SystemPromptPart
from pydantic_ai.models.test import TestModel


def show_what_it_sees(messages, info):
    """Spy model: print the roles that reach the second agent's model."""
    saved = [p.content for m in messages for p in m.parts if isinstance(p, SystemPromptPart)]
    print("system_prompt found in history:", saved)
    print("instructions sent on this run :", info.instructions)
    return ModelResponse(parts=[TextPart("ok")])


greeter = Agent(
    TestModel(),
    name="greeter",
    system_prompt="You are the greeter. Talk like a pirate.",
    instructions="Greet the customer.",
)
fixer = Agent(
    FunctionModel(show_what_it_sees),
    name="fixer",
    system_prompt="You are the fixer.",
    instructions="Fix the customer's tech problem.",
)

first = await greeter.run("Hi, my app keeps crashing.")
handed_over = await fixer.run("Can you help?", message_history=first.all_messages())

> ✅ **Checkpoint** — exactly these two lines:
>
> ```
> system_prompt found in history: ['You are the greeter. Talk like a pirate.']
> instructions sent on this run : Fix the customer's tech problem.
> ```
>
> The greeter's pirate role followed the conversation into `fixer`. Fixer's own `"You are the fixer."` was
> never sent: once you pass `message_history`, an agent's `system_prompt` is not added, whatever the history
> holds. The instructions are fixer's own, because every run works them out fresh.

Stuck with a `system_prompt` role? Give the receiving agent
`capabilities=[ReinjectSystemPrompt(replace_existing=True)]` (from `pydantic_ai.capabilities`) and its own
`system_prompt` replaces the one in the history. Docs:
[Reinject System Prompt](https://pydantic.dev/docs/ai/capabilities/reinject-system-prompt/).

When the control flow itself becomes the hard part (loops, many branches), there is
[pydantic-graph](https://pydantic.dev/docs/ai/graph/graph/), but its own docs say plain Python like `handle()`
is the shorter road for most agents.

## 🏋️ Exercise — support triage on one budget

Build `triage`, a router in front of two specialists, where every agent spends from one budget.

1. **Route and look up.** Give `triage` `deps_type=Account` (a dataclass with `name`, `plan`, `last_charge`)
   and `output_type=[to_billing, to_tech, Escalate]`. `to_billing` hands the question to a `billing_agent`
   whose plain-`def` tool `lookup_account` reads the account from `ctx.deps`. Forward `deps`, `usage` and
   `usage_limits`. Run "Why was I charged $40?" and "My password reset email never arrives." with
   `usage_limits=UsageLimits(request_limit=5)` and check each lands on the right desk.
2. **Prove it offline.** Under `override(model=FunctionModel(...))` for all three agents, `assert` that the
   billing route returns the account details, that `result.usage.requests` counts every agent, and that with
   `request_limit=2` the run raises `UsageLimitExceeded` after exactly 2 requests (pass in your own
   `RunUsage()` to count them).

*Stretch:* swap the router for a `Literal` classifier + `match`, and compare the request counts.

Solution: [`../solutions/11_support_triage.ipynb`](../solutions/11_support_triage.ipynb).

## Next →

[12 — Evals](12_evals.ipynb): prove your steering works, with test cases instead of eyeballing.